# Synaptic Routing Architecture (SRA)
## 11: 시냅스 동적 삭제(Hot-Swap 취소 및 특정 도메인 제거)

이 노트북은 SRA의 기능인 "시냅스 삭제"를 보여줍니다. 구체적으로 다음 두 가지 시나리오를 확인합니다.
1. **플러그인 제거(`pop_synapses`)**: 나중에 추가(Hot-Swap)한 시냅스를 끝에서 물리적으로 제거하고 추가 전 상태로 복원합니다.
2. **특정 도메인 제거(`clear_synapses`)**: 초기 기본 모델에서 특정 기능(예: 수학)만 추출하고 다른 사람과 공유되지 않는 시냅스를 안전하게 "제로 클리어"합니다.

※ 이 노트북은 GPU가 없어도 실행 가능합니다.

In [ ]:
# 0. 環境セットアップ (Google Colab用)
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from sra_language_models import MoESRALanguageModel
from sra_experiment import find_unshared_synapses
import torch


### 1. 플러그인 제거 (`pop_synapses`)
우선은, 소규모의 모델을 구축해, 동적으로 시냅스를 추가한 후, 그것을 `pop_synapses` 로 떼어내는 동작을 확인합니다.

In [ ]:
# モデルの初期化
dim = 128
layers = 2
num_synapses = 4
k = 2
syn_hidden = 256
vocab_size = 1000

print("=== プラグインの取り外しデモ ===")
model = MoESRALanguageModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"[初期状態] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの追加 (Hot-Swap)
print("\nプラグインとしてシナプスを 2つ 追加します...")
model.add_synapses(2, freeze_base=True)
print(f"[追加後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの取り外し (Undo Hot-Swap)
print("\n追加したシナプスを 1つ 取り外します...")
model.pop_synapses(1)
print(f"[取り外し後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

이와 같이, `pop_synapses(N)` 를 호출하는 것으로, 말미로부터 N개의 시냅스 텐서가 물리적으로 삭제되어 모델의 용량을 복원할 수 있습니다.

### 2. 특정 도메인 제거(`clear_synapses` 및 `find_unshared_synapses`)
다음으로, 복수 도메인을 학습한 베이스 모델중에서, 「특정 도메인(여기에서는 만일 `math`)에서 밖에 사용되지 않는 불필요한 시냅스」를 자동 검출해, 제로 클리어에 의해 안전하게 무효화하는 프로세스를 실증합니다.

In [ ]:
import random
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

device = "cuda" if torch.cuda.is_available() else "cpu"

# 仮のデータセットとバッチ関数の準備（デモ用）
domains = ["text", "code", "math"]
data = {d: torch.randint(0, vocab_size, (1000,)) for d in domains}

def dummy_get_batch(data_dict, batch_size, seq_len, domain):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    d_data = data_dict[domain]
    for i in range(batch_size):
        start = random.randint(0, len(d_data) - seq_len - 1)
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
    return x, y

# もう少し大きめのモデルを準備
multi_model = MoESRALanguageModel(vocab_size, dim=128, layers=2, num_synapses=16, k=2, syn_hidden=256).to(device)

In [ ]:
print("=== 特定ドメインのパージデモ ===")
print("Mathドメインでのみ使用され、他(Text, Code)と共用されていないシナプスを検索します...")

# ユーティリティを用いて対象シナプスを特定
unshared_synapses = find_unshared_synapses(
    model=multi_model, 
    data_dict=data, 
    target_domain="math", 
    other_domains=["text", "code"], 
    get_batch_func=dummy_get_batch,
    max_seq_len=32,
    threshold=0.01  # 1%以上の頻度で利用されていれば「使用している」と判定
)

print(f"\n抽出されたMath専用のシナプスインデックス: {unshared_synapses}")

※ 상기의 인덱스는, 미학습 모델의 경우는 랜덤하게 분포하기 때문에 발견되지 않는 경우가 있습니다.

추출된 인덱스(찾지 못했을 경우는 데모용으로 적당한 인덱스)를 `clear_synapses` 에 건네주어, 해당 시냅스를 제로 클리어 합니다.

In [ ]:
if len(unshared_synapses) > 0:
    target_idx = unshared_synapses[0]
else:
    print("※未学習モデルのため条件に完全合致するものがありませんでした。デモとしてシナプス 3 を対象にします。")
    unshared_synapses = [3]
    target_idx = 3

# クリア前の重みのノルムを確認
pre_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
pre_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア前] シナプス {target_idx} の Router埋め込みノルム: {pre_emb_norm:.4f}, W1ノルム: {pre_w1_norm:.4f}")

# ゼロクリア実行
multi_model.clear_synapses(unshared_synapses)
print("\nゼロクリアを実行しました。\n")

# クリア後の重みのノルムを確認
post_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
post_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア後] シナプス {target_idx} の Router埋め込みノルム: {post_emb_norm:.4f}, W1ノルム: {post_w1_norm:.4f}")

### 결론
`clear_synapses` 는 지정된 인덱스의 가중치를 물리적으로 삭제하지 않고 `0.0` 에 클리어 합니다.
이렇게 하면 다른 시냅스의 인덱스(ID)가 어긋나는 것을 방지하고 기존 메타데이터 마스크와의 호환성을 유지하면서 불필요한 계산 경로를 완전히 비활성화할 수 있습니다. 빈 슬롯이 된 인덱스에는 나중에 새 플러그인을 덮어 쓸 수 있습니다.